
# Roughness & Friction-Slope Diagnostics — Updated
_Generated 2025-08-26 17:20:07_

This notebook implements corrected and annotated computations for:

- Manning-related diagnostics (`⟨U⟩`, `⟨h⟩`, effective *n*, etc.)
- Direct friction-slope mean `⟨S_f⟩ = ⟨n^2 h^{-4/3} U^2⟩`
- Product-of-means closures:
    - `⟨S_f⟩_prod_n2AM = ⟨n^2⟩ ⟨h⟩^{-4/3} ⟨U⟩^2`
    - `⟨S_f⟩_prod_n2GM = gmean(n^2) ⟨h⟩^{-4/3} ⟨U⟩^2`
    - `⟨S_f⟩_prod_nbar2 = ⟨n⟩^2 ⟨h⟩^{-4/3} ⟨U⟩^2`
- Reynolds-averaged (RA) expansion components, with **distributed prefactor** and **exact partition**:
    - Distributed-prefactor terms use `⟨n^2 h^{-4/3}⟩` to avoid magnitude bias.
    - Exact partition retains spatial weighting and sums exactly to `⟨S_f⟩_direct`.

**Key fixes included here:**
- Avoid in-place mutation / dtype truncation on `veg` masks (cast to float + copy).
- Replace hard-coded `dx=2` with `sim.dx` where applicable.
- Use `g = 9.81` m/s².
- Guard against `h ≤ 0` to avoid explosive `h^{-4/3}`.
- Disambiguate **direct** vs **closure** notations via consistent naming.
- Safer time-difference indexing for `d⟨U⟩/dt`.


In [ ]:

import numpy as np
import pandas as pd
from scipy.stats import gmean
import warnings
import math

# No seaborn; use matplotlib only if plotting.
import matplotlib.pyplot as plt

# Constants
g = 9.81  # m/s^2

# If you rely on any unit conversion like 3.6e5 in your code, define it here for clarity.
# TODO: verify the physical meaning of this factor in your workflow.
P_CONV = 3.6e5


In [ ]:

def nanmean(x, **kwargs):
    return np.nanmean(np.asarray(x), **kwargs)

def safe_float_array(x):
    arr = np.asarray(x, dtype=float)
    return arr.copy()

def floor_depths(h, floor=0.0):
    """Mask non-physical (<=floor) depths to NaN to avoid h**(-4/3) blow-ups."""
    h = np.asarray(h, dtype=float)
    h[h <= floor] = np.nan
    return h

def central_diff_x(arr, dx):
    """Central difference along last axis (x), conserving shape (drops endpoints)."""
    arr = np.asarray(arr, dtype=float)
    return (arr[..., 2:] - arr[..., :-2]) / (2.0 * dx)

def mean_over_axes(arr, axes):
    a = np.asarray(arr, dtype=float)
    return a.mean(axis=axes)


In [ ]:

def compute_prefactors(a, h_tr):
    """Return prefactors needed for closures."""
    n2_direct = nanmean(a**2 * h_tr**(-4/3))      # ⟨n^2 h^{-4/3}⟩ (direct prefactor)
    n2_AM     = nanmean(a**2)                     # ⟨n^2⟩
    hbar_inv  = nanmean(h_tr)**(-4/3)             # ⟨h⟩^{-4/3}
    return n2_direct, n2_AM, hbar_inv

def compute_exact_partition(a, h_tr, v_tr):
    """Exact decomposition that sums to ⟨S_f⟩_direct by construction."""
    # Means and fluctuations
    U_bar = nanmean(v_tr)
    Up = v_tr - U_bar

    w = a**2 * h_tr**(-4/3)              # pointwise weight
    Sf_direct = nanmean(w * (v_tr**2))   # ⟨w U^2⟩

    Sf_U2_exact   = nanmean(w) * (U_bar**2)
    Sf_cross_exact= 2.0 * U_bar * nanmean(w * Up)
    Sf_Up2_exact  = nanmean(w * (Up**2))
    Sf_exact_sum  = Sf_U2_exact + Sf_cross_exact + Sf_Up2_exact

    return {
        '<Sf>_direct': Sf_direct,
        '<Sf>_U2(exact)': Sf_U2_exact,
        '<Sf>_cross(exact)': Sf_cross_exact,
        '<Sf>_Up2(exact)': Sf_Up2_exact,
        '<Sf>_exact_sum': Sf_exact_sum
    }

def compute_distributed_RA(a, h_tr, v_tr, use_direct_prefactor=True):
    """Distributed closure components:
    If use_direct_prefactor=True, use ⟨n^2 h^{-4/3}⟩; else use ⟨n^2⟩⟨h⟩^{-4/3}.
    """
    U_bar = nanmean(v_tr)
    h_bar = nanmean(h_tr)
    Up = v_tr - U_bar
    hp = h_tr - h_bar

    n2_direct, n2_AM, hbar_inv = compute_prefactors(a, h_tr)
    pref = n2_direct if use_direct_prefactor else (n2_AM * hbar_inv)

    term_U2  = pref * (U_bar**2)
    term_Up2 = pref * nanmean(Up**2)
    term_C1  = pref * (-(8.0/3.0) * U_bar * nanmean(Up*hp) / h_bar)
    term_C2  = pref * (-(4.0/3.0) * nanmean((Up**2)*hp) / h_bar)

    approx_sum = term_U2 + term_Up2 + term_C1 + term_C2

    return {
        '<n2 h^-4/3>': n2_direct,
        '<n2><h>^-4/3': n2_AM * hbar_inv,
        '<pref_ratio>': (n2_AM * hbar_inv) / n2_direct if np.isfinite(n2_direct) and n2_direct != 0 else np.nan,
        '<Sf>_U2': term_U2,
        '<Sf>_Up2': term_Up2,
        '<Sf>_C1': term_C1,
        '<Sf>_C2': term_C2,
        '<Sf>_approx': approx_sum
    }


In [ ]:

def compute_metrics_for_key(sim):
    """Compute all summary metrics for one row-like `sim` (e.g., a pandas Series).
    This function **does not** modify `sim` in place; it returns a dict of metrics.
    It anticipates fields like: veg (0/1), alpha_v, alpha_b, hc, vc, i_tr, So, p, l, dx, dt, Ks_v.
    """
    out = {}

    # --- Roughness field a (avoid int truncation) ---
    veg = np.asarray(sim.veg)
    a = veg.astype(float).copy()
    a[veg == 1] = float(sim.alpha_v)
    a[veg == 0] = float(sim.alpha_b)

    # --- Time-slice ---
    h_tr = np.asarray(sim.hc).copy()[sim.i_tr]
    v_tr = np.asarray(sim.vc).copy()[sim.i_tr]

    # Guard against h <= 0
    h_tr = floor_depths(h_tr, floor=0.0)

    # --- "n" related (note: `n` below is NOT a physically consistent n; kept for continuity) ---
    out['n'] = nanmean(h_tr**(1/6) * a**0.5)  # diagnostic statistic; not a true n
    out['n_tr'] = nanmean(h_tr**(2/3)) * (sim.So**0.5) / nanmean(v_tr)

    # --- Drag coefficient Cd ---
    out['Cd'] = g * nanmean(h_tr) * sim.So / nanmean(v_tr**2)

    # --- Direct & closure Sf ---
    out['<a>']  = float(nanmean(a))
    out['<a2>'] = float(nanmean(a**2))
    out['<a>2'] = float(nanmean(a))**2
    out['a_gmean'] = float(gmean(a.ravel())) if np.isfinite(a).all() else np.nan

    Sf_direct = nanmean(a**2 * h_tr**(-4/3) * (v_tr**2))
    out['<Sf>_direct'] = float(Sf_direct)
    out['<Sf>_prod_n2AM']  = float(nanmean(a**2) * (nanmean(h_tr))**(-4/3) * (nanmean(v_tr)**2))
    out['<Sf>_prod_n2GM']  = float(gmean(a.ravel()**2) * (nanmean(h_tr))**(-4/3) * (nanmean(v_tr)**2))
    out['<Sf>_prod_nbar2'] = float((nanmean(a))**2 * (nanmean(h_tr))**(-4/3) * (nanmean(v_tr)**2))

    # --- RA distributed components (use direct prefactor by default to avoid magnitude bias) ---
    ra = compute_distributed_RA(a, h_tr, v_tr, use_direct_prefactor=True)
    out.update(ra)

    # --- Exact partition (sums exactly to <Sf>_direct) ---
    exact = compute_exact_partition(a, h_tr, v_tr)
    out.update(exact)

    # --- Residuals / diagnostics ---
    out['<Sf>_resid(approx-direct)'] = out['<Sf>_approx'] - out['<Sf>_direct']
    out['<Sf>_resid(exact-direct)']  = out['<Sf>_exact_sum'] - out['<Sf>_direct']

    # Central differences (use sim.dx)
    try:
        dUdx = central_diff_x(sim.vc[sim.i_tr], sim.dx)  # shape (..., x-2)
        out['<dU/dx>'] = float(np.nanmean(dUdx))
        out['<U dU/dx>'] = float(np.nanmean(sim.vc[sim.i_tr, :, 1:-1] * dUdx))
        out['<U><dU/dx>'] = float(np.nanmean(sim.vc[sim.i_tr, :, 1:-1]) * np.nanmean(dUdx))
    except Exception as e:
        warnings.warn(f"central diff failed: {e}")
        out['<dU/dx>'] = np.nan
        out['<U dU/dx>'] = np.nan
        out['<U><dU/dx>'] = np.nan

    # Fluctuation moments
    U_bar = nanmean(v_tr)
    h_bar = nanmean(h_tr)
    Up = v_tr - U_bar
    hp = h_tr - h_bar

    out['<U>'] = float(U_bar)
    out['<h>'] = float(h_bar)
    out['<U>/<h>'] = float(U_bar / h_bar) if h_bar not in (0, np.nan) else np.nan
    out['<Up2>'] = float(nanmean(Up**2))
    out['<Up hp>'] = float(nanmean(Up * hp))
    out['<Up2 hp>'] = float(nanmean((Up**2) * hp))

    # Time derivative of <U>
    try:
        Um = mean_over_axes(sim.vc, axes=(1,2))  # mean over y,x
        dUdt = np.diff(Um) / float(sim['dt'])
        idx = int(sim.i_tr) - 1
        out['d<U>/dt'] = float(dUdt[idx])
    except Exception as e:
        warnings.warn(f"d<U>/dt failed: {e}")
        out['d<U>/dt'] = np.nan

    # Optional: integrals across (if available). Replace dx=sim.dx instead of literal 2.
    try:
        Va = integrate_U_across(a, 0.66, sim.So, sim.p - sim.Ks_v, dx=sim.dx)
        ha = integrate_h_across(a, 0.66, sim.So, sim.p - sim.Ks_v, dx=sim.dx)
        out['<Ua>'] = float(np.mean(Va / sim.l))
        out['<ha>'] = float(np.mean(ha / sim.l))
        out['<na>'] = float((np.mean(ha / sim.l))**(2/3) / np.mean(Va / sim.l) * sim.So**0.5)
    except NameError:
        # If integrate_* functions are not defined in your environment, skip gracefully.
        out['<Ua>'] = np.nan
        out['<ha>'] = np.nan
        out['<na>'] = np.nan

    return out


In [ ]:

def update_summary_with_metrics(summary):
    """Iterate rows of `summary` (index are your keys), compute metrics, and add columns."""
    # Prepare all columns that may be written
    columns = set()
    previews = {}

    # First pass: compute metrics into dicts
    metrics_per_key = {}
    for key in summary.index:
        sim = summary.loc[key]
        met = compute_metrics_for_key(sim)
        metrics_per_key[key] = met
        columns.update(met.keys())

    # Add columns to summary if missing
    for col in sorted(columns):
        if col not in summary.columns:
            summary[col] = np.nan

    # Second pass: assign values
    for key, met in metrics_per_key.items():
        for col, val in met.items():
            summary.at[key, col] = val

    return summary


In [ ]:

# --- OPTIONAL DEMO ---
# This creates a tiny synthetic `summary` if none exists, so you can see columns populate.
if 'summary' not in globals():
    ny, nx = 5, 50
    T = 10
    rng = np.random.default_rng(42)

    fake = {
        'veg': (rng.random((ny, nx)) > 0.5).astype(int),
        'alpha_v': 0.045,
        'alpha_b': 0.03,
        'hc': rng.random((T, ny, nx)) * 0.2 + 0.05,    # depths 0.05..0.25
        'vc': rng.random((T, ny, nx)) * 0.4 + 0.1,     # velocities 0.1..0.5
        'i_tr': T//2,
        'So': 1e-3,
        'p': 1.0,
        'Ks_v': 0.0,
        'l': 10.0,
        'dx': 1.0,
        'dt': 60.0
    }
    summary = pd.DataFrame([fake], index=['demo'])

# Run the update
summary = update_summary_with_metrics(summary)
display_cols = [c for c in summary.columns if c.startswith('<Sf>_') or c in ['<Sf>_direct','<n2 h^-4/3>','<n2><h>^-4/3','<pref_ratio>']]
print('Updated columns (subset):')
print(display_cols)
summary[display_cols].head(3)


In [ ]:

# Simple bar comparison for the first key (if matplotlib display is desired)
first_key = summary.index[0]
row = summary.loc[first_key]

labels = ['<Sf>_direct','<Sf>_approx','<Sf>_U2','<Sf>_Up2','<Sf>_C1','<Sf>_C2','<Sf>_exact_sum']
values = [row.get(k, np.nan) for k in labels]

plt.figure(figsize=(9,4))
plt.bar(labels, values)
plt.xticks(rotation=45, ha='right')
plt.title(f'Friction-slope components — {first_key}')
plt.ylabel('Value')
plt.tight_layout()
plt.show()
